# Busca de parâmetros do Dyna-Q

Executar antes de alterar os parâmetros fixos do notebook principal. Todas as candidatas usam 10 sementes e 600 episódios. As seeds 100–109 ficam reservadas à avaliação final. A busca sequencial no Colab pode levar alguns minutos.

In [1]:
%pip -q install gymnasium==1.2.3 numpy==2.2.6 pandas==2.3.3 numba==0.61.2 matplotlib==3.10.7 seaborn==0.13.2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.2.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ==============================================================================
# TAREFA A: FUNÇÕES DE DISCRETIZAÇÃO
# ==============================================================================

import gymnasium as gym
from gymnasium import spaces
import numpy as np

POS_MINIMA = -1.2
POS_MAXIMA = 0.6
VEL_MINIMA = -0.07
VEL_MAXIMA = 0.07

# Discretização por Grid Uniforme => classe para transformar o espaço contínuo de estados
# do Mountain Car em um grid com tamanho de linhas e de colunas passadas por parâmetros
# 1) Discretização grosseira: 10 posições diferentes e 10 velocidades possíveis
# 2) Discretização fina: 30 posições diferentes e 30 velocidades possíveis
# (passar como parâmetros)

class StateDiscretizer:
    def __init__(self, n_posicoes, n_velocidades):
        # calcula o tamanho do bloco ocupado por cada eixo dado o n
        
        self.tamanho_degrau_pos = (POS_MAXIMA - POS_MINIMA) / n_posicoes
        self.tamanho_degrau_vel = (VEL_MAXIMA - VEL_MINIMA) / n_velocidades

        self.n_posicoes = n_posicoes
        self.n_velocidades = n_velocidades

    def discretize(self, state):
        pos, vel = state
        # calcula a posição e a velocidade discretizada atual, dado a posição e a velocidade contínua
        # para não estourar o array, precisa usar o min e o max
        pos_discretizada = max(0, min(self.n_posicoes - 1, int((pos - POS_MINIMA) / self.tamanho_degrau_pos)))
        vel_discretizada = max(0, min(self.n_velocidades - 1, int((vel - VEL_MINIMA) / self.tamanho_degrau_vel)))
        return pos_discretizada, vel_discretizada

In [3]:
import numpy as np
from numba import njit


@njit
def _iniciar_rng(seed):
    np.random.seed(seed)


@njit
def _escolher_acao(q, estado, epsilon):
    if np.random.random() < epsilon:
        return np.random.randint(q.shape[1])
    # Desempate aleatório evita favorecer sempre a primeira ação.
    candidatas = np.flatnonzero(q[estado] == q[estado].max())
    return candidatas[np.random.randint(len(candidatas))]


@njit
def _atualizar(q, s, a, r, proximo, terminal, alpha, gamma):
    alvo = r if terminal else r + gamma * q[proximo].max()
    q[s, a] += alpha * (alvo - q[s, a])


@njit
def _aprender_planejar(q, s, a, r, proximo, terminal, alpha, gamma,
                      n, recompensas, proximos, terminais, vistos, pares, quantidade):
    _atualizar(q, s, a, r, proximo, terminal, alpha, gamma)
    par = s * q.shape[1] + a
    if not vistos[par]:
        vistos[par] = True
        pares[quantidade] = par
        quantidade += 1
    # Modelo clássico de última transição observada, incluindo terminalidade.
    recompensas[par] = r
    proximos[par] = proximo
    terminais[par] = terminal
    for _ in range(n):
        sim = pares[np.random.randint(quantidade)]
        ss, aa = sim // q.shape[1], sim % q.shape[1]
        _atualizar(q, ss, aa, recompensas[sim], proximos[sim],
                   terminais[sim], alpha, gamma)
    return quantidade


class DynaQAgent:
    """Uma atualização real + N atualizações de planejamento por interação."""

    def __init__(self, discretizer, n_actions=3, alpha=0.1, gamma=0.99,
                 epsilon_initial=1.0, epsilon_min=0.05, epsilon_decay=0.99,
                 planning_steps=5, seed=0):
        if not isinstance(planning_steps, (int, np.integer)) or planning_steps < 0:
            raise ValueError('planning_steps deve ser um inteiro não negativo.')
        if not (0 < alpha <= 1 and 0 <= gamma <= 1
                and 0 <= epsilon_min <= epsilon_initial <= 1
                and 0 < epsilon_decay <= 1):
            raise ValueError('Hiperparâmetros fora dos intervalos permitidos.')
        self.discretizer = discretizer
        self.alpha, self.gamma = alpha, gamma
        self.epsilon_initial, self.epsilon_min = epsilon_initial, epsilon_min
        self.epsilon_decay, self.planning_steps = epsilon_decay, planning_steps
        n_states = discretizer.n_posicoes * discretizer.n_velocidades
        self.q = np.zeros((n_states, n_actions), dtype=np.float64)
        tamanho = n_states * n_actions
        self.model_rewards = np.zeros(tamanho)
        self.model_next_states = np.zeros(tamanho, dtype=np.int64)
        self.model_terminals = np.zeros(tamanho, dtype=np.bool_)
        self.seen = np.zeros(tamanho, dtype=np.bool_)
        self.observed_pairs = np.zeros(tamanho, dtype=np.int64)
        self.n_observed = 0
        self.real_updates = self.planning_updates = 0
        # Um agente por execução/processo: o RNG compilado é reiniciado por seed.
        _iniciar_rng(seed)

    def state_index(self, observation):
        pos, vel = self.discretizer.discretize(observation)
        return pos * self.discretizer.n_velocidades + vel

    def epsilon(self, episode):
        return max(self.epsilon_min, self.epsilon_initial * self.epsilon_decay ** episode)

    def act(self, state, episode=0, greedy=False):
        return int(_escolher_acao(self.q, state, 0.0 if greedy else self.epsilon(episode)))

    def learn(self, state, action, reward, next_state, terminated):
        self.n_observed = _aprender_planejar(
            self.q, state, action, reward, next_state, terminated,
            self.alpha, self.gamma, self.planning_steps, self.model_rewards,
            self.model_next_states, self.model_terminals, self.seen,
            self.observed_pairs, self.n_observed)
        self.real_updates += 1
        self.planning_updates += self.planning_steps

    @property
    def q_updates(self):
        return self.real_updates + self.planning_updates


In [4]:
import json
import platform
import time
from pathlib import Path
from importlib.metadata import version
import pandas as pd
import gymnasium as gym

TRAIN_EPISODES = 600
SELECTION_SEEDS = list(range(10))
EVALUATION_SEEDS = list(range(100, 110))
TAIL_EPISODES = 100
RESULTS_DIR = Path('resultados_dyna_q')


def candidate_configs():
    # Busca limitada e explícita; não representa um ótimo global.
    # Dois perfis conjuntos de taxa de aprendizado, desconto e exploração.
    profiles = [dict(alpha=0.1, gamma=0.99, epsilon_decay=0.99),
                dict(alpha=0.3, gamma=0.999, epsilon_decay=0.995)]
    return [dict(config_id=f'g{grid}_p{p}_n{n}', grid=grid,
                 planning_steps=n, epsilon_initial=1.0, epsilon_min=0.05, **profile)
            for grid in (10, 30) for p, profile in enumerate(profiles)
            for n in (0, 5, 20)]


def train_one(config, seed, episodes=TRAIN_EPISODES, phase='avaliacao'):
    env = gym.make('MountainCar-v0')
    params = {k: v for k, v in config.items() if k not in ('grid', 'config_id')}
    agent = DynaQAgent(StateDiscretizer(config['grid'], config['grid']), seed=seed, **params)
    env.action_space.seed(seed)
    # Compila os kernels antes da medição; não altera Q nem consome o RNG do treino.
    _escolher_acao(agent.q, 0, 0.0)
    _aprender_planejar(agent.q.copy(), 0, 0, -1.0, 0, False,
                      agent.alpha, agent.gamma, 0,
                      agent.model_rewards.copy(), agent.model_next_states.copy(),
                      agent.model_terminals.copy(), agent.seen.copy(),
                      agent.observed_pairs.copy(), 0)
    _iniciar_rng(seed)
    total_steps = 0
    records = []
    start = time.perf_counter()
    try:
        for episode in range(episodes):
            observation, _ = env.reset(seed=seed if episode == 0 else None)
            state = agent.state_index(observation)
            reward_sum, episode_steps = 0.0, 0
            while True:
                action = agent.act(state, episode)
                observation, reward, terminated, truncated, _ = env.step(action)
                next_state = agent.state_index(observation)
                # Truncamento encerra o episódio, mas preserva bootstrap em Q.
                agent.learn(state, action, reward, next_state, bool(terminated))
                state = next_state
                reward_sum += reward
                episode_steps += 1
                total_steps += 1
                if terminated or truncated:
                    break
            records.append(dict(
                phase=phase, algorithm='Dyna-Q', **config, seed=seed,
                episode=episode + 1, episode_reward=reward_sum,
                episode_steps=episode_steps, cumulative_steps=total_steps,
                q_updates=agent.q_updates, real_q_updates=agent.real_updates,
                planning_q_updates=agent.planning_updates,
                elapsed_seconds=time.perf_counter() - start,
                epsilon=agent.epsilon(episode), success=bool(terminated)))
    finally:
        env.close()
    assert agent.q_updates == total_steps * (1 + config['planning_steps'])
    return pd.DataFrame(records), agent.q.copy()


def summarize(data):
    # Cada seed tem o mesmo peso. Não tratar episódios como réplicas independentes.
    tail = data[data.episode > TRAIN_EPISODES - TAIL_EPISODES]
    by_seed = tail.groupby(['config_id', 'grid', 'seed'], as_index=False).agg(
        final_reward=('episode_reward', 'mean'), success_rate=('success', 'mean'))
    costs = data.groupby(['config_id', 'seed'], as_index=False).agg(
        total_q_updates=('q_updates', 'max'), total_steps=('cumulative_steps', 'max'))
    by_seed = by_seed.merge(costs, on=['config_id', 'seed'])
    result = by_seed.groupby(['config_id', 'grid'], as_index=False).agg(
        mean_final_reward=('final_reward', 'mean'),
        std_final_reward=('final_reward', 'std'),
        mean_success_rate=('success_rate', 'mean'),
        mean_q_updates=('total_q_updates', 'mean'), seeds=('seed', 'nunique'))
    return result.sort_values(['mean_final_reward', 'mean_q_updates', 'config_id'],
                              ascending=[False, True, True])


def select_best(selection_data):
    ranking = summarize(selection_data)
    # Melhor configuração por resolução; avaliação final em seeds nunca usadas na busca.
    ids = ranking.drop_duplicates('grid').config_id
    return [c for c in candidate_configs() if c['config_id'] in set(ids)]


def save_metadata(directory):
    metadata = dict(environment='MountainCar-v0', episodes=TRAIN_EPISODES,
                    tail_episodes=TAIL_EPISODES, selection_seeds=SELECTION_SEEDS,
                    evaluation_seeds=EVALUATION_SEEDS, candidates=candidate_configs(),
                    selection_criterion='Maior recompensa média nos últimos 100 episódios; '
                    'desempate pelo menor número médio de atualizações Q.',
                    python=platform.python_version(),
                    packages={p: version(p) for p in
                              ['numpy', 'gymnasium', 'numba', 'pandas', 'matplotlib', 'seaborn']})
    (directory / 'metadados.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
save_metadata(RESULTS_DIR)
frames = []
for config in candidate_configs():
    for seed in SELECTION_SEEDS:
        frame, _ = train_one(config, seed, phase='selecao')
        frames.append(frame)
        frame.to_csv(RESULTS_DIR / f"busca_{config['config_id']}_seed{seed}.csv", index=False)
    print('Concluído:', config['config_id'])
selection = pd.concat(frames, ignore_index=True)
selection.to_csv(RESULTS_DIR / 'busca.csv', index=False)
ranking = summarize(selection)
ranking.to_csv(RESULTS_DIR / 'ranking_busca.csv', index=False)
best = select_best(selection)
(RESULTS_DIR / 'melhores_parametros.json').write_text(json.dumps(best, indent=2), encoding='utf-8')
display(ranking)
print(json.dumps(best, indent=2))
